<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/05-agents/06-harness-engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harness Engineering

**Goal:** Name the discipline this whole section has been teaching piece by piece — engineering the *scaffold around* the model call — and add the three levers it hasn't yet made explicit: context assembly, tool-result shaping, and verification loops.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## The third axis

You now have three ways to change what an LLM system does, and they're independent:

- **Prompt engineering** — the *message* you send. (Section 01, woven throughout.)
- **Model adaptation** — the *weights* behind the call. (Section 06: prompt → RAG → fine-tune.)
- **Harness engineering** — the *scaffold around* the call: how you assemble context each turn, how you shape what tools return, the loop and its stopping conditions, and the checks that verify the model's work.

Most of the reliability of a shipped agent comes from the third one, and it's the one job titles rarely name. Here's the thing: **section 05 has been teaching harness engineering the whole time** — just one component at a time.

| You built | Harness component |
|---|---|
| [01 Agent loop](01-agent-loop-from-scratch.ipynb) | the loop — the skeleton the model runs inside |
| [02 Tool design](02-tool-design.ipynb) | the tools' *input* side — what the model can call |
| [03 Guardrails & budgets](03-guardrails-and-budgets.ipynb) | stopping conditions, bounds, loop detection |
| [04 MCP](04-mcp-and-the-tool-ecosystem.ipynb) | connecting *external* tools into the harness |
| [05 Skills](05-skills-and-progressive-disclosure.ipynb) | packaging know-how the harness loads on demand |

This notebook names the discipline and fills the three gaps the section left implicit:

1. **Context assembly** — what actually goes into the window *each turn* (and what to drop when it grows).
2. **Tool-result shaping** — what you feed *back* in after a tool runs (the return side of tool design).
3. **Verification loops** — the harness checking the model's work and looping on failure, not just on tool calls.

> **⭐ Key takeaway —** the model is fixed; the harness is yours. When an agent misbehaves, reach for the scaffold — context, tool results, the loop, the checks — before you reach for a bigger model or a fine-tune.

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client — the only dependency this notebook needs.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

In [ ]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

## The same tools, the same loop

We reuse section 01's fake filesystem, its four tools, and a compact version of the loop — so every change below is visibly a change to the *harness*, not the task. Nothing here is new; skim it and move on.

In [ ]:
import json, re

# Section 01's fake filesystem: path -> content. Safe, inspectable, resettable.
FS = {
    "notes/monday.md":    "Team lunch: $42.50\nTaxi to client site: $18.00\n",
    "notes/tuesday.md":   "Conference tickets, 2 x $150.00 each\n",
    "notes/wednesday.md": "Cloud credits top-up: $75.25\nCoffee for the workshop: $23.10\n",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def calculator(expression):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {"list_files": list_files, "read_file": read_file,
         "write_file": write_file, "calculator": calculator}

def execute_tool(name, args):
    fn = IMPLS.get(name)
    if fn is None:
        return f"Error: unknown tool '{name}'."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"Error: bad arguments for {name}: {e}"

def _tool(name, desc, props, required):
    return {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object", "properties": props, "required": required}}}

TOOLS = [
    _tool("list_files", "List every file path in the workspace, one per line.", {}, []),
    _tool("read_file", "Read the full contents of one file.",
          {"path": {"type": "string"}}, ["path"]),
    _tool("write_file", "Create or overwrite a file with the given content.",
          {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"]),
    _tool("calculator", "Evaluate an arithmetic expression with + - * / and parentheses.",
          {"expression": {"type": "string"}}, ["expression"]),
]

def run(messages, tools=TOOLS, shape_result=lambda name, out: out, max_turns=8):
    """Section 01's loop, compacted, with ONE new seam: shape_result lets the
    harness transform each tool's output before it re-enters the context."""
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        choice = resp.choices[0]
        msg = choice.message
        if choice.finish_reason != "tool_calls":
            return msg.content or "", messages
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            out = shape_result(tc.function.name, execute_tool(tc.function.name, args))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return "[hit max_turns]", messages

print("scaffold ready:", list(IMPLS))

## Lever 1 — Context assembly: what goes in the window *each turn*

The API is stateless. Every turn re-sends the **entire** `messages` list — that's the observation notebooks 01 and 03 kept flagging ("per-turn cost rises because you re-send the whole history"). So the context window isn't a place the model *has*; it's a thing your harness *rebuilds and pays for on every call*.

That reframing hands you a lever. You decide what survives into the next turn. On a long run — dozens of tool calls, large file reads — the raw transcript blows past the window and the bill climbs turn over turn. The harness move is **compaction**: once the transcript crosses a threshold, replace the old middle with a short summary and keep only the head (the task) and the tail (recent turns).

The cell below measures a transcript, then compacts it — no model call needed to see the point.

In [ ]:
def transcript_size(messages):
    """Rough token proxy: characters/4. Good enough to see the trend."""
    chars = sum(len(json.dumps(m, default=str)) for m in messages)
    return len(messages), chars, chars // 4

def compact(messages, keep_recent=4, summary=None):
    """Keep the first message (the task) + the last `keep_recent`; replace the
    middle with one summary line. A real harness would ask the model to write
    the summary; here we synthesize it so the mechanism is visible and free."""
    if len(messages) <= keep_recent + 1:
        return messages
    dropped = messages[1:-keep_recent]
    note = summary or f"[{len(dropped)} earlier turns compacted: files read and totals computed so far]"
    return [messages[0], {"role": "user", "content": note}, *messages[-keep_recent:]]

# Simulate a long run: the task, then many bulky tool exchanges.
long_run = [{"role": "user", "content": "Audit every notes file and reconcile the totals."}]
for i in range(20):
    long_run.append({"role": "assistant", "content": None,
                     "tool_calls": [{"id": f"c{i}", "type": "function",
                        "function": {"name": "read_file", "arguments": '{"path":"notes/x.md"}'}}]})
    long_run.append({"role": "tool", "tool_call_id": f"c{i}",
                     "content": "Some line item: $" + "1234.56, " * 20})  # bulky result

before = transcript_size(long_run)
after  = transcript_size(compact(long_run))
print(f"before compaction: {before[0]:>3} msgs, ~{before[2]:>5} tokens")
print(f"after  compaction: {after[0]:>3} msgs, ~{after[2]:>5} tokens")
print(f"→ {100 * (before[2]-after[2]) // before[2]}% smaller, task + recent turns preserved")

> **⚠️ Production reality —** compaction is lossy on purpose, and *what* you drop is a design decision. Summarize the wrong middle and the agent forgets a constraint it agreed to 15 turns ago. Keep the task, keep recent turns, and keep anything the model committed to (a plan, a running total) — summarize the rest. This is the same "context is a budget" idea from [05 Skills](05-skills-and-progressive-disclosure.ipynb), applied to the transcript instead of to loaded know-how.

## Lever 2 — Tool-result shaping: what you feed *back* in

Notebook 02 designed the tools' **input** side — names, descriptions, schemas the model reads before calling. This is the **output** side: what your harness does with the result *before it becomes the next turn's context*. The model never sees the raw return value; it sees whatever you append as the `tool` message. Two shaping moves earn their keep:

- **Truncate big blobs.** A tool that dumps a 50 KB file poisons every subsequent turn (you re-send it each time — Lever 1). Head + tail + an honest "N chars omitted" marker keeps the signal and caps the cost.
- **Make errors actionable.** An error the model can *act on* ("no file at 'x'; call list_files") turns a dead turn into a recovery. A raw stack trace or a bare `None` just burns turns. Section 01's tools already do this; shaping is where you enforce it centrally.

The `run()` loop above already has the seam — the `shape_result` hook. Below we pass one in and watch a giant result get capped.

In [ ]:
# A tool result far too big to re-send every turn.
FS["notes/dump.md"] = "line item $9.99\n" * 400  # ~6 KB

def shape(name, out, limit=300):
    """Harness-level shaping applied to EVERY tool result before it re-enters context."""
    if len(out) <= limit:
        return out
    head, tail = out[:limit // 2], out[-limit // 2:]
    return f"{head}\n...[{len(out) - limit} chars omitted by the harness]...\n{tail}"

raw = read_file("notes/dump.md")
shaped = shape("read_file", raw)
print(f"raw result:    {len(raw):>5} chars  (re-sent every turn if unshaped)")
print(f"shaped result: {len(shaped):>5} chars\n")
print(shaped[:220], "...")

# Run the real agent with shaping wired in — the giant file no longer bloats context.
FS.pop("summary.md", None)
answer, _ = run(
    [{"role": "user", "content": "Read notes/dump.md and tell me roughly how many line items it has."}],
    shape_result=shape)
print("\n=== answer ===\n", answer)

## Lever 3 — Verification loops: the harness checks the work

Notebook 01's loop stops on one condition: *the model didn't ask for a tool.* But "the model declared itself done" is not "the work is correct." The harness can hold a higher bar — run a **check** against the result and, on failure, feed the specific problem back as a new turn so the model fixes it. This is [03's trajectory-eval](03-guardrails-and-budgets.ipynb) idea pointed at the *output*: an assertion the harness owns, not a hope the model got it right.

The task: total the expenses and write `summary.md`. The check the harness enforces: **the number in `summary.md` must actually equal the sum of the expenses.** If it doesn't, the harness says exactly what's wrong and loops — bounded, of course (Lever from 03: every loop needs a cap).

In [ ]:
def truth_total():
    """The harness's own ground truth — computed in code, not trusted from the model."""
    amounts = [float(x) for f, c in FS.items() if f.startswith("notes/") and f != "notes/dump.md"
               for x in re.findall(r"\$?([0-9]+\.[0-9]{2})", c)]
    # tuesday.md says "2 x $150.00" — the regex sees one 150.00, so add the second.
    return round(sum(amounts) + 150.00, 2)

def written_total():
    """Pull the number the model wrote into summary.md, or None if absent."""
    m = re.findall(r"([0-9]+\.[0-9]{2})", FS.get("summary.md", ""))
    return float(m[-1]) if m else None

def run_verified(task, max_repairs=2):
    messages = [{"role": "user", "content": task}]
    for attempt in range(max_repairs + 1):
        answer, messages = run(messages, shape_result=shape)
        want, got = truth_total(), written_total()
        if got is not None and abs(got - want) < 0.01:
            return f"verified after {attempt} repair(s): total {got} is correct"
        # Verification failed — feed the SPECIFIC gap back as the next turn.
        messages.append({"role": "user", "content":
            f"That's not right. The verified total is {want:.2f} but summary.md "
            f"{'has ' + format(got, '.2f') if got is not None else 'has no total'}. "
            "Recompute with the calculator and rewrite summary.md."})
    return f"still wrong after {max_repairs} repairs (wanted {want}, got {written_total()})"

FS.pop("summary.md", None)
print(run_verified(
    "Read every notes/*.md file (ignore notes/dump.md), add up all expenses with the "
    "calculator, and write summary.md stating the total. Note: 'notes/tuesday.md' lists "
    "'2 x $150.00' — that's two items."))
print("summary.md:", FS.get("summary.md", "(not written)").strip())

> **🚩 Common mistake —** trusting the model's own "done." The model saying *"I've written the total to summary.md"* is a claim, not a guarantee — it may have fat-fingered the arithmetic or written the wrong number. A harness that computes ground truth in code and loops until it matches is the difference between a demo and something you'd let run unattended. Verify in code what you can; reserve the [04 LLM-as-judge](../04-evals/02-llm-as-judge.ipynb) approach for what you can't.

## The harness is the unit you engineer

Step back and the section reads as one object. An agent is not "a smart model" — it's a **model plus a harness**, and every reliability property lives in the harness:

```
task ─▶ ┌────────── HARNESS — your code (the model is fixed) ──────────┐ ─▶ result
        │                                                              │
        │  (1) assemble        ┌────────────┐        (2) shape         │
        │      context ──────▶ │ MODEL call │ ──────▶ tool results ─┐  │
        │         ▲            └────────────┘                       │  │
        │         │                                          (3) verify│
        │         └───── loop: bounded + guarded ◀──────────────────┘  │
        │               (nb 01 loop · nb 03 bounds)                    │
        │                                                              │
        └──────────────────────────────────────────────────────────────┘

  (1) context assembly    (2) result shaping    (3) verification loop
```

The model in the middle is fixed. Everything else — what you put in front of it, what you feed back, when you stop, what you check — is code you own and test. That's why "harness engineering" is worth naming: when the system is flaky, the fix is almost always a harness change (tighter context, shaped results, a real verification step, a better bound), *not* a bigger model. Reach for the weights (Section 06) only after the scaffold is right.

## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Treat the context window as a per-turn budget you rebuild and pay for | Assume the model "remembers" — and let the transcript grow unbounded |
| Compact long transcripts: keep the task + recent turns + commitments, summarize the middle | Truncate blindly (drop the oldest N), losing a constraint the agent agreed to |
| Shape every tool result before it re-enters context — cap size, keep head+tail | Feed raw 50 KB blobs back and re-send them every turn |
| Return errors the model can act on ("no such file; call list_files") | Surface bare stack traces or `None` and burn a turn |
| Verify the output in code and loop on failure — bounded | Trust the model's own "done"; ship the first right-looking answer |
| Compute ground truth in the harness, not from the model's claim | Ask the model whether its own answer is correct and believe it |
| When flaky, fix the harness first (context, results, checks, bounds) | Reach for a bigger model or a fine-tune before the scaffold is right |

## Where the frameworks come in — LangChain & LangGraph

Notebook 01 already named what agent frameworks add on top of the raw loop (persistence, observability, handoffs, operational plumbing). Now that you've also built tool design, guardrails, MCP, skills, and the harness, here's how the **RAG/agent framework** names on job posts map onto what you built — so the resume keyword and the understanding travel together:

- **LangChain** — the components layer: LLM wrappers, tool/`@tool` definitions, prompt templates, memory. Its `AgentExecutor` *is* the section-05 loop; its tool abstraction *is* your tool schema + dispatch (notebook 02).
- **LangGraph** — the orchestration layer: your loop expressed as an explicit **state graph** (nodes = steps, edges = transitions), with checkpointing and human-in-the-loop pauses built in. It's what you reach for when the harness (notebook 06) — context assembly, verification loops, bounded retries — needs to be *persisted and resumable*, not just in-memory.
- **LlamaIndex** — overlaps on the RAG side (see the section-03 bridge); its agents wire retrieval-first workflows.

> **⭐ Key takeaway —** these frameworks run the *same* while-loop you wrote in ~40 lines. They add persistence, tracing, and multi-agent wiring — never a new kind of magic. Having hand-built the loop, its guardrails, and its harness, you can read a LangGraph diagram and see your own code, and judge on exact terms: what does it persist, what does it show me, what does it hide?

**When to reach for one:** durable/resumable runs, a team that needs shared traces, or multi-agent handoffs (the systems view: [Agent Orchestration walkthrough](https://www.calm.rocks/resources/prepare-interview/system-design/agent-orchestration-walkthrough/)). **When not:** a bounded, single-purpose agent is often clearer and cheaper as the explicit loop — and recall the section-03/05 discipline that a *pipeline* beats an agent when the steps are known ([guardrails](03-guardrails-and-budgets.ipynb)).

> **🔵 Interview signal —** the framework-free path was never anti-framework. It's what lets you say "I use LangGraph, and I can tell you which of its features I actually need and which I'd skip" — the judgment a senior AI engineer is hired for.

## Exercises

1. **Model-written summaries.** Replace the synthetic string in `compact()` with a real model call: when the transcript crosses the threshold, ask the model (no tools, small `max_tokens`) to summarize the dropped middle in two sentences, and splice *that* in. Compare token savings and whether the agent still finishes the task correctly.
2. **A shaping policy per tool.** `shape()` treats every tool the same. Make it a dict of per-tool shapers — e.g. `read_file` gets head+tail truncation, `list_files` is never truncated, `calculator` is passed through. Show a task where the uniform limit hurt and the per-tool policy fixes it.
3. **Verify the trajectory too.** `run_verified` checks the *output*. Add a second check on the *path* (from [03's trajectory eval](03-guardrails-and-budgets.ipynb)): fail and repair if the agent wrote `summary.md` without ever calling the `calculator`. Confirm both checks must pass before it returns `verified`.
4. **Break it on purpose.** Set `max_repairs=0` and change `truth_total()` to expect the wrong number. Watch the harness report a clean, specific failure instead of silently accepting bad output — then write one sentence on why "fail loud with the gap" beats "return whatever the model last wrote."